<h2>Imports: </h2>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [35]:
forecasting_horizon = 10
input_chunk_length = 180
ticker = 'NVDA'
interval = "minute"
interval_range = 15
days = 100

<h2>The NEWS SENTIMENTAL analysis functions:</h2>

In [2]:
def extract_article_text(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Extract all paragraphs
            paragraphs = soup.find_all('p')
            article_text = ' '.join([p.get_text(strip=True) for p in paragraphs])

            if not article_text.strip():
                return "Could not extract text. Might be paywalled."
            
            return article_text
        else:
            return f"Failed with status code {response.status_code}"
    
    except requests.exceptions.RequestException as e:
        return f"Request failed: {e}"

In [3]:
import requests
import pandas as pd

def fetch_polygon_news(ticker, api_key='LTikapZvdZjraWfVP2r_QgAvTX_oZSZw'):
    """
    Fetches the latest news articles for a given stock ticker using Polygon.io's News API and extracts the article text,
    filtering out articles from The Motley Fool (i.e. URLs containing "fool.com").
    """
    base_url = 'https://api.polygon.io/v2/reference/news'
    params = {
        'ticker': ticker,
        'limit': 100,
        'order': 'descending',
        'sort': 'published_utc',
        'apiKey': api_key
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        articles = data.get('results', [])
        
        # Extract relevant information, filtering out articles from fool.com
        news_data = []
        for article in articles:
            url = article.get('article_url', '')
            # Skip articles with URLs that contain "fool.com"
            if url and "fool.com" in url.lower():
                continue
            
            title = article.get('title', 'No Title')
            published = article.get('published_utc', 'Unknown Date')
            # Here extract_article_text is assumed to be defined elsewhere.
            text = extract_article_text(url) if url else None
            
            news_data.append({
                'Title': title,
                'Published': published,
                'URL': url,
                'Text': text
            })
        
        return pd.DataFrame(news_data)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching news: {e}")
        return pd.DataFrame()


In [4]:
df = fetch_polygon_news('AAPL')
df['Published'] = pd.to_datetime(df['Published'], errors='coerce').dt.strftime('%Y-%m-%d')
df

,Title,Published,URL,Text
0,"Apple, MicroStrategy are Among the Most Shorte...",2025-03-21,https://www.investing.com/analysis/apple-micro...,Technology stocks were heavily shorted by hedg...
1,How The Smart Money Gets Rich When Markets Crash,2025-03-19,https://www.benzinga.com/news/large-cap/25/03/...,Benzinga Rankings give you vital metrics on an...
2,Purpose Investments Inc. annonce les distribut...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 19:15 ET| Source:Investissement..."
3,"Equity Exposure Plummets, Cash Holdings Jump A...",2025-03-18,https://www.benzinga.com/economics/macro-econo...,Benzinga Rankings give you vital metrics on an...
4,Online Gaming Market Forecast Report and Compa...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 07:00 ET| Source:Research and M..."
5,Apple Stock: Optimism Rides High Despite 2025 ...,2025-03-17,https://www.investing.com/analysis/apple-stock...,Apple Inc (NASDAQ:AAPL) is currently navigatin...
6,Consumer Tech News (Mar 10-Mar 14): Manus AI C...,2025-03-16,https://www.benzinga.com/news/large-cap/25/03/...,Benzinga Rankings give you vital metrics on an...
7,"Coinbase, Apple And Robinhood Are Among Top La...",2025-03-16,https://www.benzinga.com/news/large-cap/25/03/...,Benzinga Rankings give you vital metrics on an...
8,Global Market for Micro and Mini LEDs 2025-203...,2025-03-13,https://www.globenewswire.com/news-release/202...,"March 13, 2025 11:57 ET| Source:Research and M..."
9,"Apple's AR Glasses Approach Aside, Google Make...",2025-03-12,https://www.benzinga.com/25/03/44267530/while-...,Benzinga Rankings give you vital metrics on an...


In [5]:
df['Text'].iloc[0]

'Technology stocks were heavily shorted by hedge fund and alternative investment managers last month, according to Hazeltree, a financial data firm. Hazeltree’s February 2025 Shortside Crowdedness Report listed Apple (NASDAQ:AAPL), Micron (NASDAQ:MU), and Super Micro Computer (NASDAQ:SMCI) among the top 5 most crowded, or shorted, large-cap U.S. stocks. Further, 8 of the 10 most shorted stocks were from the technology sector. In developing its monthly list, Hazeltree examines the securities that are being shorted by the highest percentage of funds. It tracks 15,000 global equities, sourced from anonymized data within 700 asset manager funds. The stocks are graded on a scale of 1-99, with 99 representing the stock that the highest percentage of funds are shorting. In February, an energy stock, Chevron (NYSE:CVX), was actually the most shorted large-cap name, with a crowdedness score of 99. Super Micro Computer, which has been dealing with internal controls issues and the threat of being

<h4>Testing the NEWS SENTIMENTAL analysis</h4>

In [6]:
import pandas as pd
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# Load FinBERT for financial sentiment analysis
MODEL_NAME = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


Device set to use cpu


In [7]:

def drcr_sentiment_analysis(article_text):
    """
    Apply Dual Reverse Chain Reasoning (DRCR) for implicit sentiment analysis.
    Only the first 500 words of the article text are used.
    Returns a dictionary with:
      - 'dummy': the sentiment label ("positive", "negative", or "neutral")
      - 'score': the corresponding confidence score
    """
    # Truncate the text to the first 500 words
    words = article_text.split()[:500]
    truncated_text = ' '.join(words)
    
    # Use the sentiment pipeline with truncation (max 512 tokens)
    positive_reasoning = sentiment_pipeline(
        f"Considering a positive market sentiment: {truncated_text}",
        truncation=True,
        max_length=512
    )[0]
    negative_reasoning = sentiment_pipeline(
        f"Considering a negative market sentiment: {truncated_text}",
        truncation=True,
        max_length=512
    )[0]
    
    # Retrieve labels in lowercase for consistency
    pos_label = positive_reasoning['label'].lower()
    neg_label = negative_reasoning['label'].lower()
    
    # Calculate adjusted confidence scores
    pos_score = positive_reasoning['score'] if pos_label == 'positive' else 1 - positive_reasoning['score']
    neg_score = negative_reasoning['score'] if neg_label == 'negative' else 1 - negative_reasoning['score']
    
    # Determine final sentiment based on contrastive reasoning
    if pos_score > neg_score:
        return {"dummy": "positive", "score": pos_score}
    elif neg_score > pos_score:
        return {"dummy": "negative", "score": neg_score}
    else:
        return {"dummy": "neutral", "score": (pos_score + neg_score) / 2}

# Assuming your DataFrame 'df' has a column 'Text' with the article text.
# Apply the sentiment analysis function and store the result in a new column 'result'
df['result'] = df['Text'].apply(drcr_sentiment_analysis)

# Expand the dictionary into two new columns: 'dummy' and 'score'
df[['dummy', 'score']] = df['result'].apply(pd.Series)

# Optionally, drop the 'result' column if no longer needed
df = df.drop(columns=['result'])

print(df[['dummy', 'score']])


       dummy     score
0   negative  0.335413
1   positive  0.088053
2   positive  0.076616
3   negative  0.974132
4   positive  0.746416
5   positive  0.859860
6   negative  0.731539
7   negative  0.061053
8   positive  0.692235
9   negative  0.620232
10  negative  0.795692
11  positive  0.365667
12  positive  0.675926
13  positive  0.907656
14  negative  0.903807
15  positive  0.375580
16  positive  0.513139
17  positive  0.373670
18  negative  0.515831
19  negative  0.108535
20  positive  0.457676
21  negative  0.971934
22  negative  0.872640


In [8]:
df

,Title,Published,URL,Text,dummy,score
0,"Apple, MicroStrategy are Among the Most Shorte...",2025-03-21,https://www.investing.com/analysis/apple-micro...,Technology stocks were heavily shorted by hedg...,negative,0.335413
1,How The Smart Money Gets Rich When Markets Crash,2025-03-19,https://www.benzinga.com/news/large-cap/25/03/...,Benzinga Rankings give you vital metrics on an...,positive,0.088053
2,Purpose Investments Inc. annonce les distribut...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 19:15 ET| Source:Investissement...",positive,0.076616
3,"Equity Exposure Plummets, Cash Holdings Jump A...",2025-03-18,https://www.benzinga.com/economics/macro-econo...,Benzinga Rankings give you vital metrics on an...,negative,0.974132
4,Online Gaming Market Forecast Report and Compa...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 07:00 ET| Source:Research and M...",positive,0.746416
5,Apple Stock: Optimism Rides High Despite 2025 ...,2025-03-17,https://www.investing.com/analysis/apple-stock...,Apple Inc (NASDAQ:AAPL) is currently navigatin...,positive,0.859860
6,Consumer Tech News (Mar 10-Mar 14): Manus AI C...,2025-03-16,https://www.benzinga.com/news/large-cap/25/03/...,Benzinga Rankings give you vital metrics on an...,negative,0.731539
7,"Coinbase, Apple And Robinhood Are Among Top La...",2025-03-16,https://www.benzinga.com/news/large-cap/25/03/...,Benzinga Rankings give you vital metrics on an...,negative,0.061053
8,Global Market for Micro and Mini LEDs 2025-203...,2025-03-13,https://www.globenewswire.com/news-release/202...,"March 13, 2025 11:57 ET| Source:Research and M...",positive,0.692235
9,"Apple's AR Glasses Approach Aside, Google Make...",2025-03-12,https://www.benzinga.com/25/03/44267530/while-...,Benzinga Rankings give you vital metrics on an...,negative,0.620232


In [9]:
def drcr_sentiment(ticker, api_key = 'LTikapZvdZjraWfVP2r_QgAvTX_oZSZw'):
    df = fetch_polygon_news(ticker, api_key)
    df['Published'] = pd.to_datetime(df['Published'], errors='coerce').dt.strftime('%Y-%m-%d')
    df['result'] = df['Text'].apply(drcr_sentiment_analysis)
    # Expand the dictionary into two new columns: 'dummy' and 'score'
    df[['dummy', 'score']] = df['result'].apply(pd.Series)
    df = df.drop(columns=['result'])
    return df

In [10]:
df_t = drcr_sentiment('NVDA')

In [11]:
df_t

,Title,Published,URL,Text,dummy,score
0,"ROSEN, SKILLED INVESTOR COUNSEL, Encourages Mo...",2025-03-20,https://www.globenewswire.com/news-release/202...,"March 20, 2025 17:43 ET| Source:The Rosen Law ...",positive,0.089434
1,MongoDB Stock at a Buy the Dip Moment Despite ...,2025-03-20,https://www.investing.com/analysis/mongodb-sto...,Scalable database provider MongoDB (NASDAQ:MDB...,positive,0.368805
2,SoftServe Wins NVIDIA’s 2025 Americas NPN Serv...,2025-03-19,https://www.globenewswire.com/news-release/202...,"March 19, 2025 11:03 ET| Source:SoftServe, Inc...",positive,0.870665
3,What's Going On With Super Micro Computer Stoc...,2025-03-19,https://www.benzinga.com/news/events/25/03/443...,Benzinga Rankings give you vital metrics on an...,positive,0.800743
4,"NVIDIA Teams Up with GE, Google, IBM, And Crow...",2025-03-19,https://www.benzinga.com/25/03/44393619/nvidia...,Benzinga Rankings give you vital metrics on an...,positive,0.604449
5,"General Atomics, UC San Diego Collaborate to L...",2025-03-19,https://www.globenewswire.com/news-release/202...,"March 18, 2025 20:10 ET| Source:General Atomic...",positive,0.673547
6,Purpose Investments Inc. annonce les distribut...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 19:15 ET| Source:Investissement...",positive,0.076616
7,MPWR Lead Plaintiff Deadline Approaching – Con...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 17:54 ET| Source:Robbins LLPRob...",negative,0.782056
8,Cirrascale Cloud Services Debuts Next-Generati...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 15:00 ET| Source:Cirrascale Clo...",positive,0.495811
9,NVIDIA Announces DGX Spark and DGX Station Per...,2025-03-18,https://www.globenewswire.com/news-release/202...,"March 18, 2025 14:59 ET| Source:NVIDIANVIDIA S...",positive,0.145460


<h2>INSIDER TRADES and different static data building</h2>

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

import os
import re
import joblib


import time
import requests
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

In [13]:
forecasting_horizon = 10
input_chunk_length = 180
ticker = 'NVDA'
interval = "minute"
interval_range = 15
days = 100

In [14]:
from news import scrape_insider_data, scrape_congress_trading_data, sentiment, get_zacks_rank, is_nyse_open
from fundamental import fundamental
from technical import technical, get_stock_open_prices, technical_polygon, technical_indicators

In [15]:
insider_df = pd.DataFrame()
congress_df = pd.DataFrame()
sentiment_df = pd.DataFrame()
zacks_rank_df = pd.DataFrame()
fundamental_df = pd.DataFrame()
technical_df = pd.DataFrame()

os.makedirs(f"test_data\{ticker}", exist_ok=True)

insider_df = scrape_insider_data(ticker)
print('congress')
congress_df = scrape_congress_trading_data(ticker)
print('sentiment')
sentiment_df = sentiment([ticker])
print('zacks')
zacks_rank = get_zacks_rank(ticker)
zacks_rank_df = pd.DataFrame({"Zacks_Rank": [zacks_rank]})
print('fundamental')
fundamental_score_df = fundamental(ticker)
# Save each dataframe to CSV for further use
insider_df.to_csv(f"test_data\{ticker}\{ticker}_Insider_Trading.csv", index=False)
print('congress')
congress_df.to_csv(f"test_data\{ticker}\{ticker}_Congress_Trades.csv", index=False)
print('sentiment')
sentiment_df.to_csv(f"test_data\{ticker}\{ticker}_Sentiment_Data.csv", index=False)
print('zacks')
zacks_rank_df.to_csv(f"test_data\{ticker}\{ticker}_Zacks_Rank.csv", index=False)
print('fundamental')
fundamental_score_df.to_csv(f"test_data\{ticker}\{ticker}_Fundamental_Score.csv", index=False)

congress
sentiment



Device set to use cpu


Sentiment Report saved as sentiment_report-21-03-2025.xlsx
zacks
fundamental
Fetching financial data for NVDA...
congress
sentiment
zacks
fundamental


C:\TWS API\source\pythonclient\test_data\test_1_data\fundamental.py:47: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill').fillna(method='bfill')  # Fill missing values


<h4>Introducting the NEWS SENTIMENTAL to our static data</h4>

In [16]:
def extract_article_text(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Extract all paragraphs
            paragraphs = soup.find_all('p')
            article_text = ' '.join([p.get_text(strip=True) for p in paragraphs])

            if not article_text.strip():
                return "Could not extract text. Might be paywalled."
            
            return article_text
        else:
            return f"Failed with status code {response.status_code}"
    
    except requests.exceptions.RequestException as e:
        return f"Request failed: {e}"

In [17]:
def fetch_polygon_news(ticker, api_key='LTikapZvdZjraWfVP2r_QgAvTX_oZSZw'):
    """
    Fetches the latest news articles for a given stock ticker using Polygon.io's News API and extracts the article text,
    filtering out articles from The Motley Fool (i.e. URLs containing "fool.com").
    """
    base_url = 'https://api.polygon.io/v2/reference/news'
    params = {
        'ticker': ticker,
        'limit': 100,
        'order': 'descending',
        'sort': 'published_utc',
        'apiKey': api_key
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        articles = data.get('results', [])
        
        # Extract relevant information, filtering out articles from fool.com
        news_data = []
        for article in articles:
            url = article.get('article_url', '')
            # Skip articles with URLs that contain "fool.com"
            if url and "fool.com" in url.lower():
                continue
            
            title = article.get('title', 'No Title')
            published = article.get('published_utc', 'Unknown Date')
            # Here extract_article_text is assumed to be defined elsewhere.
            text = extract_article_text(url) if url else None
            
            news_data.append({
                'Title': title,
                'Published': published,
                'URL': url,
                'Text': text
            })
        
        return pd.DataFrame(news_data)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching news: {e}")
        return pd.DataFrame()


<h5>The functiing that puts the sentimental part together:</h5>

In [18]:
def drcr_sentiment(ticker, api_key = 'LTikapZvdZjraWfVP2r_QgAvTX_oZSZw'):
    df = fetch_polygon_news(ticker, api_key)
    df['Published'] = pd.to_datetime(df['Published'], errors='coerce').dt.strftime('%Y-%m-%d')
    df['result'] = df['Text'].apply(drcr_sentiment_analysis)
    # Expand the dictionary into two new columns: 'dummy' and 'score'
    df[['dummy', 'score']] = df['result'].apply(pd.Series)
    df = df.drop(columns=['result'])
    return df

<h3>Putting together the static data & running into a lot of problems</h3>

In [19]:
import pandas as pd
import re

# Load data
insider_df = pd.read_csv(f"test_data/{ticker}/{ticker}_Insider_Trading.csv")
congress_df = pd.read_csv(f"test_data/{ticker}/{ticker}_Congress_Trades.csv")

# Convert date columns to standardized YYYY-MM-DD format
date_columns = ['Disclosed (EST)', 'Date', 'Traded Date', 'Filed Date']
for col in date_columns:
    if col in insider_df.columns:
        insider_df[col] = pd.to_datetime(insider_df[col], errors='coerce').dt.strftime('%Y-%m-%d')
    if col in congress_df.columns:
        congress_df[col] = pd.to_datetime(congress_df[col], errors='coerce').dt.strftime('%Y-%m-%d')

# Rename columns for consistency
congress_df.rename(columns={'Politician Name': 'Name', 
                            'Politician Position': 'Title', 
                            'Filed Date': 'Disclosed (EST)'}, inplace=True)

# Extract and convert transaction amount to numeric
if 'Transaction Amount' in congress_df.columns:
    congress_df['Transaction Amount'] = congress_df['Transaction Amount'].apply(
        lambda x: int(re.search(r'\$(\d[\d,]*)$', x).group(1).replace(',', '')) 
        if isinstance(x, str) and re.search(r'\$(\d[\d,]*)$', x) else None
    )

# Fetch open prices efficiently
if 'Date' in insider_df.columns:
    traded_dates_list = insider_df['Date'].tolist()
    open_prices = get_stock_open_prices(ticker, traded_dates_list)
    insider_df['Open Price'] = open_prices

# Ensure proper renaming after open price retrieval
insider_df.rename(columns={'Purchase/Sale': 'Transaction Type', 
                           'Politician Position': 'Title', 
                           'Date': 'Traded Date'}, inplace=True)

# Handle missing 'Shares' column gracefully
if 'Shares' in insider_df.columns:
    insider_df['Shares'] = pd.to_numeric(insider_df['Shares'], errors='coerce')
    insider_df['Transaction Amount'] = insider_df['Shares'] * insider_df['Open Price']
else:
    insider_df['Transaction Amount'] = None  # Default to None if Shares column is missing

# Merge DataFrames
merged_df = pd.concat([congress_df, insider_df], ignore_index=True)

# Drop unnecessary columns if they exist
merged_df.drop(columns=['Stock', 'Description', 'Shares', 'Open Price'], errors='ignore', inplace=True)

# Sort for easier grouping
merged_df_sorted = merged_df.sort_values(by=['Name', 'Title', 'Traded Date', 'Transaction Type'])

# Initialize list to store combined rows
combined_rows = []
current_group = None
total_amount = 0

# Iterate and combine rows based on Name, Title, Traded Date, and Transaction Type
for _, row in merged_df_sorted.iterrows():
    if current_group and (current_group == (row['Name'], row['Title'], row['Traded Date'], row['Transaction Type'])):
        total_amount += row['Transaction Amount']  # Sum the transaction amount
        last_disclosed = row['Disclosed (EST)']  # Update last disclosed date
    else:
        # Save the previous group and reset for the new group
        if current_group:
            combined_rows.append({
                'Name': current_group[0],
                'Title': current_group[1],
                'Traded Date': current_group[2],
                'Transaction Type': current_group[3],
                'Transaction Amount': total_amount,
                'Disclosed (EST)': last_disclosed
            })

        # Start a new group
        current_group = (row['Name'], row['Title'], row['Traded Date'], row['Transaction Type'])
        total_amount = row['Transaction Amount']
        last_disclosed = row['Disclosed (EST)']

# Append the last group after the loop ends
if current_group:
    combined_rows.append({
        'Name': current_group[0],
        'Title': current_group[1],
        'Traded Date': current_group[2],
        'Transaction Type': current_group[3],
        'Transaction Amount': total_amount,
        'Disclosed (EST)': last_disclosed
    })

# Create DataFrame from combined rows
insider_trades_df = pd.DataFrame(combined_rows)

# Ensure 'Traded Date' is datetime
insider_trades_df['Traded Date'] = pd.to_datetime(insider_trades_df['Traded Date'])

# Sort & Reset Index
insider_trades_df = insider_trades_df.sort_values(by='Traded Date').reset_index(drop=True)


C:\Users\david\AppData\Local\Temp\ipykernel_10940\3609251873.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  insider_df[col] = pd.to_datetime(insider_df[col], errors='coerce').dt.strftime('%Y-%m-%d')


In [26]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# Load insider trading and congress trading datasets
insider_df = pd.read_csv(f"test_data/{ticker}/{ticker}_Insider_Trading.csv")
insider_df['Date'] = pd.to_datetime(insider_df['Date'], format="%b %d, %Y").dt.strftime("%Y-%m-%d")
congress_df = pd.read_csv(f"test_data/{ticker}/{ticker}_Congress_Trades.csv")

# Convert date columns to standardized YYYY-MM-DD format
date_columns = ['Disclosed (EST)', 'Date', 'Traded Date', 'Filed Date']
for col in date_columns:
    if col in insider_df.columns:
        insider_df[col] = pd.to_datetime(insider_df[col], errors='coerce').dt.strftime('%Y-%m-%d')
    if col in congress_df.columns:
        congress_df[col] = pd.to_datetime(congress_df[col], errors='coerce').dt.strftime('%Y-%m-%d')

# Rename columns for consistency
congress_df.rename(columns={
    'Politician Name': 'Name', 
    'Politician Position': 'Title', 
    'Filed Date': 'Disclosed (EST)'
}, inplace=True)

# Extract and convert transaction amount to numeric
congress_df['Transaction Amount'] = congress_df['Transaction Amount'].apply(
    lambda x: int(re.search(r'\$(\d[\d,]*)$', x).group(1).replace(',', '')) if isinstance(x, str) and re.search(r'\$(\d[\d,]*)$', x) else None
)

# ---- **Fetch Open Prices Efficiently** ----
# Get unique traded dates
traded_dates_list = insider_df['Date'].dropna().unique().tolist()

# Fetch open prices for all dates at once
open_prices = get_stock_open_prices(ticker, traded_dates_list)

# Map open prices back to the dataframe
open_price_map = dict(zip(traded_dates_list, open_prices))
insider_df['Open Price'] = insider_df['Date'].map(open_price_map)

# Rename columns for consistency
insider_df.rename(columns={'Purchase/Sale': 'Transaction Type', 'Date': 'Traded Date'}, inplace=True)

# Convert Shares to numeric and calculate Transaction Amount
insider_df['Shares'] = pd.to_numeric(insider_df['Shares'], errors='coerce').fillna(0)
insider_df['Open Price'] = pd.to_numeric(insider_df['Open Price'], errors='coerce').fillna(0)
insider_df['Transaction Amount'] = insider_df['Shares'] * insider_df['Open Price']

# Drop unnecessary columns
insider_df.drop(columns=['Stock', 'Description', 'Shares'], errors='ignore', inplace=True)

# ---- **Merge Insider & Congress DataFrames** ----
merged_df = pd.concat([congress_df, insider_df], ignore_index=True)

# Sort the merged DataFrame for easier grouping
merged_df_sorted = merged_df.sort_values(by=['Name', 'Title', 'Traded Date', 'Transaction Type'])

# ---- **Combine Trades from Same Person on Same Date** ----
combined_rows = []
current_group = None
total_amount = 0

# Iterate and combine rows based on Name, Title, Traded Date, and Transaction Type
for _, row in merged_df_sorted.iterrows():
    if current_group and (current_group == (row['Name'], row['Title'], row['Traded Date'], row['Transaction Type'])):
        total_amount += row['Transaction Amount']  # Sum the transaction amount
        last_disclosed = row['Disclosed (EST)']  # Update last disclosed date
    else:
        # Save the previous group and reset for the new group
        if current_group:
            combined_rows.append({
                'Name': current_group[0],
                'Title': current_group[1],
                'Traded Date': current_group[2],
                'Transaction Type': current_group[3],
                'Transaction Amount': total_amount,
                'Disclosed (EST)': last_disclosed
            })

        # Start a new group
        current_group = (row['Name'], row['Title'], row['Traded Date'], row['Transaction Type'])
        total_amount = row['Transaction Amount']
        last_disclosed = row['Disclosed (EST)']

# Append the last group after the loop ends
if current_group:
    combined_rows.append({
        'Name': current_group[0],
        'Title': current_group[1],
        'Traded Date': current_group[2],
        'Transaction Type': current_group[3],
        'Transaction Amount': total_amount,
        'Disclosed (EST)': last_disclosed
    })

# ---- **Create Final Cleaned DataFrame** ----
insider_trades_df = pd.DataFrame(combined_rows)

# Ensure 'Traded Date' is datetime format and sort
insider_trades_df['Traded Date'] = pd.to_datetime(insider_trades_df['Traded Date'])
insider_trades_df = insider_trades_df.sort_values(by='Traded Date').reset_index(drop=True)

C:\Users\david\AppData\Local\Temp\ipykernel_10940\827376526.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  insider_df[col] = pd.to_datetime(insider_df[col], errors='coerce').dt.strftime('%Y-%m-%d')


In [27]:
static_df = insider_trades_df
# Define the cutoff as the last 30 days from today
latest_sentiment_score = sentiment_df['general_sentiment_score'].iloc[0]
latest_zacks_score = zacks_rank

recent_cutoff = datetime.now() - timedelta(days=30)

# Convert 'Traded Date' to datetime format if not already
static_df["Traded Date"] = pd.to_datetime(static_df["Traded Date"])

# Assign latest scores only to recent transactions
static_df["Zacks Score"] = static_df["Traded Date"].apply(
    lambda x: latest_zacks_score if x >= recent_cutoff else None
)

static_df["Sentiment Score"] = static_df["Traded Date"].apply(
    lambda x: latest_sentiment_score if x >= recent_cutoff else None
)
static_df.to_csv(f"test_data\{ticker}\{ticker}_Static_Data.csv", index=False)

In [28]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from datetime import datetime

# Drop 'Zacks Score' and 'Sentiment Score' if they exist
static_df = static_df.drop(columns=['Zacks Score', 'Sentiment Score'], errors='ignore')

# One-hot encoding for 'Transaction Type' and 'Name'
static_df = pd.get_dummies(static_df, columns=['Transaction Type'], dtype=int)
static_df = pd.get_dummies(static_df, columns=['Name'], dtype=int)

# Encode 'Title' using Label Encoding
title_encoder = LabelEncoder()
if "Title" in static_df.columns:
    static_df['Title'] = title_encoder.fit_transform(static_df['Title'])
    static_df['Title'] = static_df['Title'].astype(int)

# Convert date columns to datetime format (only if they exist)
if "Traded Date" in static_df.columns:
    static_df["Traded Date"] = pd.to_datetime(static_df["Traded Date"], errors="coerce")
if "Disclosed (EST)" in static_df.columns:
    static_df["Disclosed (EST)"] = pd.to_datetime(static_df["Disclosed (EST)"], errors="coerce")

# Calculate disclosure delay (only if both columns exist)
if "Traded Date" in static_df.columns and "Disclosed (EST)" in static_df.columns:
    static_df["Disclosure Delay"] = (static_df["Disclosed (EST)"] - static_df["Traded Date"]).dt.days

# Handle missing 'Quarter_score' column
if "Quarter_score" in static_df.columns and static_df["Quarter_score"].notna().any():
    static_df["Quarter_score"] = static_df["Quarter_score"].astype(str).str.extract(r'(\d+)/\d+')
    static_df["Quarter_score"] = pd.to_numeric(static_df["Quarter_score"], errors='coerce').astype('Int64')
else:
    # If 'Quarter_score' is missing or empty, filter data to include only the current and last year
    if "Traded Date" in static_df.columns:
        current_year = datetime.now().year
        static_df = static_df[static_df["Traded Date"].dt.year >= current_year - 1]
        print("'Quarter_score' is missing or empty. Using only data from the current and last year.")
    else:
        print("'Traded Date' is missing. Skipping year-based filtering.")

# Drop date columns only after all processing is done
static_df = static_df.drop(columns=["Disclosed (EST)"], errors="ignore")

# Convert boolean columns to integers
bool_cols = static_df.select_dtypes(include=bool).columns
static_df[bool_cols] = static_df[bool_cols].astype(int)
static_df = static_df.reset_index(drop=True)
print("Data processing completed successfully.")


'Quarter_score' is missing or empty. Using only data from the current and last year.
Data processing completed successfully.


In [30]:
static_df['Traded Date']

0    2024-01-02
1    2024-02-08
2    2024-02-09
3    2024-02-20
4    2024-02-20
        ...    
69   2025-02-26
70   2025-02-26
71   2025-03-10
72   2025-03-13
73   2025-03-17
Name: Traded Date, Length: 74, dtype: datetime64[ns]

In [24]:
static_covariates = pd.DataFrame(static_df.mean()).T

In [36]:

def assign_news_score(static_df, news_df, tolerance='7D'):
    """
    Assigns a news sentiment score to each row in static_df based on the news article
    whose 'Published' date is closest to the static_df's 'Traded Date'. If no news article
    is found within the tolerance, a score of 0 is assigned.
    
    Parameters:
        static_df (pd.DataFrame): DataFrame with a 'Traded Date' column.
        news_df (pd.DataFrame): DataFrame with 'Published' (date) and 'score' (sentiment score) columns.
        tolerance (str): Maximum difference between 'Traded Date' and 'Published' date (default '7D').
        
    Returns:
        pd.DataFrame: static_df with a new column 'News Score' added.
    """
    # Convert date columns to datetime
    static_df['Traded Date'] = pd.to_datetime(static_df['Traded Date'], errors='coerce')
    news_df['Published'] = pd.to_datetime(news_df['Published'], errors='coerce')
    
    # Sort both DataFrames by their respective date columns
    static_df_sorted = static_df.sort_values('Traded Date').reset_index(drop=True)
    news_df_sorted = news_df.sort_values('Published').reset_index(drop=True)
    
    # Perform an asof merge to match each traded date with the nearest published date within tolerance
    merged = pd.merge_asof(static_df_sorted, 
                           news_df_sorted[['Published', 'score']], 
                           left_on='Traded Date', 
                           right_on='Published', 
                           direction='nearest', 
                           tolerance=pd.Timedelta(tolerance))
    
    # If no news article was found, 'score' will be NaN. Replace NaN with 0.
    merged['score'] = merged['score'].fillna(0)
    
    # Add the 'News Score' column to the static_df
    static_df_sorted['News Score'] = merged['score']
    
    return static_df_sorted

In [37]:
news_df = drcr_sentiment(ticker, api_key = 'LTikapZvdZjraWfVP2r_QgAvTX_oZSZw')

In [38]:
static_with_news = assign_news_score(static_df, news_df, tolerance='7D')

In [39]:
static_with_news

,Title,Traded Date,Transaction Amount,Transaction Type_Purchase,Transaction Type_Sale,Transaction Type_Sale (Full),Transaction Type_Sale (Partial),Name_Alan S. Lowenthal,Name_BURGESS ROBERT K,Name_Blake Moore,...,Name_STEVENS MARK A,Name_Stephen F. Lynch,"Name_Sullivan, Dan",Name_Thomas R. Suozzi,Name_Thomas Suozzi,"Name_Tuberville, Tommy","Name_Whitehouse, Sheldon","Name_Wyden, Ron",Disclosure Delay,News Score
0,7,2024-01-02,50000.00,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,24,0.000000
1,2,2024-02-08,15000.00,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,28,0.000000
2,3,2024-02-09,15000.00,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,206,0.000000
3,3,2024-02-20,600000.00,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,30,0.000000
4,3,2024-02-20,15000.00,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,21,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,7,2025-02-26,15000.00,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,8,0.000000
70,2,2025-02-26,15000.00,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,0.000000
71,8,2025-03-10,292663.70,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0.813284
72,8,2025-03-13,6240507.72,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,4,0.644270
